In [3]:
import pandas as pd

In [4]:
df_preprocessed = pd.read_parquet(r"C:\Users\griep\Git Repos\AAVA_tool\data\02_intermediate\preprocessed_data.parquet")

In [5]:
print(df_preprocessed["opleiding"].unique())

['Bachelor HBO/WO' 'Master HBO/WO' 'MBO' 'HAVO/VWO' 'VMBO'
 'Basisonderwijs']


In [9]:
import pandas as pd
import numpy as np
import re
import random

# --- 1. CONFIGURATION: Adjust these probabilities ---

# Define probabilities for demographic variables
PROBABILITIES = {
    # Harmfulness
    "harmfulness_level": {"low": 0.8, "medium": 0.15, "high": 0.05},
    
    # Gender (Focus on diversity)
    "gender": {"male": 0.45, "female": 0.45, "non-binary": 0.10},
    
    # Education Level (Based on rough Dutch distribution)
    "education_level": {'Bachelor HBO/WO': 0.30, 'Master HBO/WO': 0.20, 'MBO': 0.25, 'HAVO/VWO': 0.15, 'VMBO': 0.05, 'Basisonderwijs': 0.05},
    
    # Ethnicity (Focus on common Dutch groups)
    "ethnicity": {
        "Dutch (Native)": 0.70,
        "Turkish": 0.05,
        "Moroccan": 0.05,
        "Surinamese": 0.05,
        "Antillean": 0.03,
        "Indonesian": 0.03,
        "Other European": 0.09
    },
    
    # Nationality (Aligned with ethnicity for simplicity)
    "nationality": {
        "Dutch": 0.85,
        "Turkish": 0.05,
        "Moroccan": 0.05,
        "German": 0.03,
        "Other EU": 0.02
    }
}

# --- 2. DATA LOADING & SENTENCE SPLITTING LOGIC ---

# 1. Load the original data (df_ethnicity contains the article text in 'text')
df_ethnicity = pd.read_csv(r"C:\Users\griep\Git Repos\AAVA_tool\data\01_raw\transcript_data\df_ethnicity.csv")
df_ethnicity = df_ethnicity # Use a small subset for quick testing

# Helper function to split text into sentences
def split_text_into_sentences(text):
    """Splits an article text into a list of sentences."""
    # Regex to split by sentence enders, keeping the ender attached to the sentence
    # It looks for (.?!) followed by whitespace, but not if preceded by a decimal or title initial (like Dr.)
    sentences = re.split(r'(?<=[.?!])\s+(?=[A-Z])', text)
    return [s.strip() for s in sentences if s.strip()]

# Create the new list of data rows
new_data_rows = []

for index, row in df_ethnicity.iterrows():
    article_text = row['text']
    all_sentences = split_text_into_sentences(article_text)
    
    # Check if there are enough sentences; use all if fewer than 3
    if len(all_sentences) >= 3:
        # Select 3 random sentences from the article
        selected_sentences = random.sample(all_sentences, 3)
    else:
        selected_sentences = all_sentences

    for sentence in selected_sentences:
        new_data_rows.append({
            "sentence": sentence,
            "article": article_text  # Keep the full article for context
        })

# Create the base DataFrame
df_dummy = pd.DataFrame(new_data_rows)


# --- 3. LOGIC FOR RANDOM ASSIGNMENT WITH PROBABILITIES ---

def assign_probabilistically(df, column_name, probabilities):
    """Assigns values to a column based on defined probabilities."""
    values = list(probabilities.keys())
    probs = list(probabilities.values())
    
    # Use numpy.random.choice to select values based on weights
    df[column_name] = np.random.choice(
        values,
        size=len(df),
        p=probs
    )
    return df

# Assign Age: Random integer within a defined range
# Assuming age range of 18 to 70 for general public studies
df_dummy["age"] = np.random.randint(18, 71, size=len(df_dummy))


# Apply the probabilistic assignment to all defined categories
df_dummy = assign_probabilistically(df_dummy, "harmfulness_level", PROBABILITIES["harmfulness_level"])
df_dummy = assign_probabilistically(df_dummy, "gender", PROBABILITIES["gender"])
df_dummy = assign_probabilistically(df_dummy, "education_level", PROBABILITIES["education_level"])
df_dummy = assign_probabilistically(df_dummy, "ethnicity", PROBABILITIES["ethnicity"])
df_dummy = assign_probabilistically(df_dummy, "nationality", PROBABILITIES["nationality"])


# --- 4. FINAL CLEANUP AND OUTPUT ---

# Reorder columns to match the initial desired structure
data_columns = [
    "harmfulness_level",
    "gender",
    "age",
    "education_level",
    "ethnicity",
    "nationality",
    "sentence",
    "article",
]
df_dummy = df_dummy[data_columns]

print(f"Dummy DataFrame created with {len(df_dummy)} rows.")
# print(df_dummy.head())

# Example to save the dummy data (optional)
# df_dummy.to_csv(r"C:\Users\griep\Git Repos\AAVA_tool\data\01_raw\transcript_data\df_dummy_labeled.csv", index=False)

Dummy DataFrame created with 5562 rows.
